# Module 297 — Garak Report Reader

This notebook accompanies HTB Section 9. It explains garak's probe/detector vocabulary and calculates rates from the sample counts printed in the lesson.

It uses only Python's standard library and the HTB example numbers. It does not install garak, call a model provider, use an API key, or send any prompts.

## What the scanner does

A **probe** is a set or family of test inputs. A **detector** checks the model's responses for a behavior of interest. Garak sends the selected probes through a model interface, then reports how responses were classified.

The HTB example command selects Replicate as the hosting platform, a model identifier on that platform, and one DAN probe. HTB says external API use is optional and outside the module requirements, so this notebook does not reproduce the network scan.

The detailed JSON-lines report can contain prompts and model responses. Treat it as assessment data, protect it, and do not commit it. A hosted API may incur cost, and large scans can take time.

## Interpret the sample rate

For these HTB output lines, the lesson explains “FAIL ok on k/n” as k okay responses out of n attempts. The failure count is n minus k.

Read the formula aloud:

**failure rate equals (attempts minus okay responses) divided by attempts.**

The fraction is then multiplied by 100 to show a percentage. It estimates how often this specific probe/detector/model combination triggered a failure in this run. It is not a universal probability that an attack works.

In [ ]:
# Transcribe only counts shown on the HTB page; no prompts or responses are stored.
sample_results = [
    {
        "probe": "dan.Dan_11_0",
        "detector": "dan.DAN",
        "attempts": 5,
        "okay_responses": 0,
    },
    {
        "probe": "dan.Dan_11_0",
        "detector": "mitigation.MitigationBypass",
        "attempts": 5,
        "okay_responses": 2,
    },
    {
        "probe": "HijackHateHumansMini",
        "detector": "promptinject.AttackRogueString",
        "attempts": 500,
        "okay_responses": 145,
    },
    {
        "probe": "HijackKillHumansMini",
        "detector": "promptinject.AttackRogueString",
        "attempts": 500,
        "okay_responses": 280,
    },
    {
        "probe": "HijackLongPromptMini",
        "detector": "promptinject.AttackRogueString",
        "attempts": 500,
        "okay_responses": 140,
    },
]

for result in sample_results:
    print(result["probe"], result["detector"], result["okay_responses"], "/", result["attempts"])

In [ ]:
def calculate_failure_rate(attempts, okay_responses):
    """Return (failed attempts, failure rate) for a valid sample count."""
    if attempts <= 0:
        raise ValueError("attempts must be a positive count")
    if not 0 <= okay_responses <= attempts:
        raise ValueError("okay_responses must be between zero and attempts")

    # Attempts that were not counted as okay represent detector/model failures.
    failures = attempts - okay_responses
    # Divide by the total to get a fraction; multiply by 100 for a percentage.
    percentage = failures / attempts * 100
    return failures, percentage

for result in sample_results:
    failures, rate = calculate_failure_rate(
        result["attempts"],
        result["okay_responses"],
    )
    print(
        f"{result['probe']} | {result['detector']}: "
        f"{failures}/{result['attempts']} failures ({rate:.0f}%)"
    )

# Match the lesson's sample: 5/5, 3/5, 355/500, 220/500, 360/500.
expected_failures = [5, 3, 355, 220, 360]
actual_failures = [
    calculate_failure_rate(r["attempts"], r["okay_responses"])[0]
    for r in sample_results
]
assert actual_failures == expected_failures

## Read the numbers with care

A result depends on the model version, the selected probe, the detector, and the particular run. The lesson's DAN detector marks 5 of 5 as failures; its mitigation detector marks 3 of 5. In the 500-attempt scan, the shown failure rates are 71%, 44%, and 72%.

Different detectors can disagree because they look for different signals. A detector may produce false positives or miss a behavior. Review representative raw outputs when authorized and safe; do not rely on a single headline score as proof that a model is secure or vulnerable.

Before a real scan, define the authorized model and probe scope, attempt count, API cost limit, and handling rules for reports. The course does not require any external provider account or paid scan.